In [35]:
import torch
import numpy as np


def salvar_dataset_npz(loader, arquivo, max_samples=None):
    dados = []
    labels = []
    total_samples = 0

    for batch in loader:
        # formato (x, y)
        x, y = batch

        # PyTorch ou tensor compatível com numpy()
        if torch.is_tensor(x):
            x = x.cpu().numpy()
        elif hasattr(x, "numpy"):
            x = x.numpy()

        if torch.is_tensor(y):
            y = y.cpu().numpy()
        elif hasattr(y, "numpy"):
            y = y.numpy()

        x = np.asarray(x)
        y = np.asarray(y)

        # Quando o loader fornece uma única amostra, y vira escalar.
        # Convertendo para batch de tamanho 1, o restante do código
        # continua funcionando para Dataset e DataLoader.
        if y.ndim == 0:
            x = np.expand_dims(x, axis=0)
            y = np.expand_dims(y, axis=0)

        if max_samples is not None:
            remaining = max_samples - total_samples
            if remaining <= 0:
                break
            if x.shape[0] > remaining:
                x = x[:remaining]
                y = y[:remaining]

        dados.append(x)
        labels.append(y)
        total_samples += x.shape[0]

        if max_samples is not None and total_samples >= max_samples:
            break

    if not dados:
        raise ValueError("Nenhuma amostra foi coletada do loader.")

    dados = np.concatenate(dados, axis=0)
    labels = np.concatenate(labels, axis=0)

    np.savez_compressed(
        arquivo,
        data=dados,
        labels=labels
    )

    print(f"Arquivo '{arquivo}.npz' salvo com {len(labels)} amostras.")

# Testes Tonic

In [36]:
%pip install tonic
import tonic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import os
import tonic
import torch

bs = 128
steps = 90
# collate = tonic.collation.PadTensors(batch_first=False)
to_frame = tonic.transforms.ToFrame(
    #sensor_size=tonic.datasets.NMNIST.sensor_size, time_window=1e3
    sensor_size=tonic.datasets.NMNIST.sensor_size, n_time_bins=steps
)
test_ds = tonic.datasets.NMNIST("./nmnist", transform=to_frame, train=False)

g = torch.Generator()
g.manual_seed(42)

test_dl = torch.utils.data.DataLoader(
    #test_ds, shuffle=True, batch_size=bs, collate_fn=collate, generator=g
    test_ds, shuffle=True, batch_size=bs, generator=g
)

In [38]:
data, label = test_ds[1000]

In [39]:
print(data.shape)

(90, 2, 34, 34)


In [40]:
print(label)

1


In [41]:
salvar_dataset_npz(test_dl, f"nir_examples\\n-mnist-{steps}-steps", 1000)

Arquivo 'nir_examples\n-mnist-90-steps.npz' salvo com 1000 amostras.


# Criando Dados Pequenos

In [ ]:
import numpy as np

X_np = np.array([
    # amostra 0
    [
        [1,0,0,0,1],
        [0,1,0,0,0],
        [0,0,1,0,0],
    ],
    # amostra 1
    [
        [0,1,1,0,0],
        [0,0,1,1,0],
        [0,0,0,1,1],
    ],
    # amostra 2
    [
        [1,0,0,0,0],
        [1,0,0,0,0],
        [0,1,0,0,0],
    ],
    # amostra 3
    [
        [0,0,0,0,1],
        [0,0,0,1,0],
        [0,0,1,0,0],
    ],
], dtype=np.float32)

Y_np = np.array([0, 1, 2, 3], dtype=np.float32)

## Torch

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.from_numpy(X_np)
Y = torch.from_numpy(Y_np)

dataset = TensorDataset(X, Y)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

In [ ]:
salvar_dataset_npz(dataloader, "z_mini_torch.npz")

Arquivo 'z_mini_torch.npz.npz' salvo.


## Tensorflow

In [ ]:
import tensorflow as tf
import numpy as np

batch_size = 1

# Cria o Dataset
dataset = tf.data.Dataset.from_tensor_slices((X_np, Y_np))

# Embaralha e cria batches
dataset = dataset.batch(batch_size)

# Teste
for xb, yb in dataset.take(1):
    print(xb.shape, yb.shape)

2026-01-13 16:11:12.274925: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-13 16:11:12.304073: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-13 16:11:12.304113: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-13 16:11:12.304119: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 16:11:12.309100: I tensorflow/core/platform/cpu_feature_g

(1, 3, 5) (1,)


2026-01-13 16:11:13.721023: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-01-13 16:11:13.742342: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-01-13 16:11:13.743772: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [ ]:
salvar_dataset_npz(dataset, "z_mini_tf.npz")

Arquivo 'z_mini_tf.npz.npz' salvo.


# NeuroHLS

In [ ]:
from NeuroHls import *

In [ ]:
neuro_hls = NeuroHls("z_test")

In [ ]:
model_config = neuro_hls.get_dummy_model_config()

In [ ]:
print(model_config)

------------------------------
Layer 1: Dense (784, 128)
------------------------------

	Unroll Factors:
		- Accum: 1
		- Fire: 1
	Quantization (ap_fixed<16, 8>):
		- Total bits: 16
		- Integer bits: 8
		- Fractional bits: 8

------------------------------
Layer 2: Dense (128, 10)
------------------------------

	Unroll Factors:
		- Accum: 1
		- Fire: 1
	Quantization (ap_fixed<16, 8>):
		- Total bits: 16
		- Integer bits: 8
		- Fractional bits: 8



In [ ]:
neuro_hls.implement_model_from_config(model_config)

## Criando o Testbench

In [ ]:
neuro_hls.define_test_dataset("z_mini_torch.npz", data_is_binary=True, step_count=10, different_sample_per_step=False)

In [ ]:
neuro_hls.create_testbench(total_samples=100, batch_size=30)

Testbench Criado
